In [8]:
import sys; sys.path.append("..")
from src.data import load_matches

matches = load_matches()
print(matches.groupby("season").size())          # 380 per complete season
print(matches[["FTHG", "FTAG"]].mean())          # home ~1.5, away ~1.2
print(sorted(matches["HomeTeam"].unique()))      # eyeball team names for inconsistencies

ValueError: No objects to concatenate

In [ ]:
P

In [2]:
import sys; sys.path.append("..")
from src import data
print(data.RAW)
print(data.RAW.exists())
print(len(list(data.RAW.glob("E0_*.csv"))))

/home/nameer12345/pl-season-predictor/data/raw
True
22


In [3]:
import sys; sys.path.append("..")
from src.data import load_matches

matches = load_matches()
print(matches.groupby("season").size())
print(matches[["FTHG", "FTAG"]].mean())
print(sorted(matches["HomeTeam"].unique()))

season
0506    380
0607    380
0708    380
0809    380
0910    380
1011    380
1112    380
1213    380
1314    380
1415    380
1516    380
1617    380
1718    380
1819    380
1920    380
2021    380
2122    380
2223    380
2324    380
2425    380
2526    380
2627     50
dtype: int64
FTHG    1.542839
FTAG    1.200747
dtype: float64
['Arsenal', 'Aston Villa', 'Birmingham', 'Blackburn', 'Blackpool', 'Bolton', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Cardiff', 'Charlton', 'Chelsea', 'Coventry', 'Crystal Palace', 'Derby', 'Everton', 'Fulham', 'Huddersfield', 'Hull', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Middlesbrough', 'Newcastle', 'Norwich', "Nott'm Forest", 'Portsmouth', 'QPR', 'Reading', 'Sheffield United', 'Southampton', 'Stoke', 'Sunderland', 'Swansea', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wigan', 'Wolves']


/home/nameer12345/pl-season-predictor/notebooks/../src/data.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["season"] = f.stem.split("_")[1]
/home/nameer12345/pl-season-predictor/notebooks/../src/data.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["season"] = f.stem.split("_")[1]
/home/nameer12345/pl-season-predictor/notebooks/../src/data.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all c

In [5]:
import sys; sys.path.append("..")
from src.data import load_matches
from src.model import fit_poisson

matches = load_matches()
season = matches[matches["season"] == "2526"]
model = fit_poisson(season)
print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  goals   No. Observations:                  760
Model:                            GLM   Df Residuals:                      720
Model Family:                 Poisson   Df Model:                           39
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1072.7
Date:                Mon, 12 Oct 2026   Deviance:                       762.51
Time:                        22:41:02   Pearson chi2:                     636.
No. Iterations:                     5   Pseudo R-squ. (CS):             0.1243
Covariance Type:            nonrobust                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           

/home/nameer12345/pl-season-predictor/notebooks/../src/data.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["season"] = f.stem.split("_")[1]
/home/nameer12345/pl-season-predictor/notebooks/../src/data.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["season"] = f.stem.split("_")[1]
/home/nameer12345/pl-season-predictor/notebooks/../src/data.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all c

In [6]:
import numpy as np, pandas as pd

p = model.params
attack = p[p.index.str.startswith("team[")].rename(lambda s: s[len("team[T."):-1])
conceded = p[p.index.str.startswith("opponent[")].rename(lambda s: s[len("opponent[T."):-1])

table = pd.DataFrame({"attack": attack, "conceded": conceded})
table.loc["Arsenal"] = 0.0          # the reference team
table = np.exp(table).sort_values("attack", ascending=False)
print(table.round(2))
print("home advantage multiplier:", np.exp(p["home"]).round(3))

                attack  conceded
Man City          1.09      1.30
Arsenal           1.00      1.00
Man United        0.99      1.85
Liverpool         0.91      1.95
Bournemouth       0.84      1.98
Chelsea           0.84      1.90
Aston Villa       0.81      1.79
Brentford         0.79      1.90
Newcastle         0.77      2.00
Brighton          0.75      1.67
Leeds             0.71      2.03
Tottenham         0.70      2.07
Nott'm Forest     0.69      1.85
Fulham            0.68      1.85
Everton           0.68      1.81
West Ham          0.67      2.35
Sunderland        0.60      1.73
Crystal Palace    0.59      1.84
Burnley           0.56      2.69
Wolves            0.40      2.41
home advantage multiplier: 1.247


In [7]:
log_table = np.log(table)
log_table = log_table - log_table.mean()     # centre so the average team = 0
centred = np.exp(log_table).sort_values("attack", ascending=False)
print(centred.round(2))


                attack  conceded
Man City          1.49      0.70
Arsenal           1.36      0.54
Man United        1.35      0.99
Liverpool         1.24      1.05
Bournemouth       1.14      1.06
Chelsea           1.14      1.02
Aston Villa       1.10      0.96
Brentford         1.08      1.02
Newcastle         1.04      1.08
Brighton          1.01      0.90
Leeds             0.97      1.09
Tottenham         0.95      1.11
Nott'm Forest     0.94      0.99
Fulham            0.92      0.99
Everton           0.92      0.97
West Ham          0.91      1.26
Sunderland        0.82      0.93
Crystal Palace    0.80      0.99
Burnley           0.76      1.45
Wolves            0.54      1.30


In [8]:
centred["rating"] = centred["attack"] / centred["conceded"]
print(centred.sort_values("rating", ascending=False).round(2))

                attack  conceded  rating
Arsenal           1.36      0.54    2.53
Man City          1.49      0.70    2.12
Man United        1.35      0.99    1.36
Liverpool         1.24      1.05    1.18
Aston Villa       1.10      0.96    1.14
Brighton          1.01      0.90    1.13
Chelsea           1.14      1.02    1.11
Bournemouth       1.14      1.06    1.08
Brentford         1.08      1.02    1.06
Newcastle         1.04      1.08    0.97
Nott'm Forest     0.94      0.99    0.95
Everton           0.92      0.97    0.95
Fulham            0.92      0.99    0.93
Leeds             0.97      1.09    0.89
Sunderland        0.82      0.93    0.88
Tottenham         0.95      1.11    0.85
Crystal Palace    0.80      0.99    0.82
West Ham          0.91      1.26    0.72
Burnley           0.76      1.45    0.53
Wolves            0.54      1.30    0.42
